# Virtue of Complexity — данные (часть 1)
**Kelly, Malamud, Zhou (2024), Journal of Finance 79(1).** Раздел V.A, сноски 33–34.

Этот ноутбук готовит данные для всей команды:
1. скачивает месячный датасет Goyal–Welch с сайта Amit Goyal;
2. строит **15 предикторов** из сноски 33: `dfy, infl, svar, de, lty, tms, tbl, dfr, dp, dy, ltr, ep, b/m, ntis` + лаг рыночной доходности;
3. строит цель — **excess return CRSP value-weighted** (`CRSP_SPvw − Rfree`);
4. стандартизирует всё **только по прошлому** (никакого look-ahead):
   * доходность делится на корень из среднего квадрата за прошлые 12 месяцев (uncentered std, сноска 34);
   * каждый предиктор делится на expanding std по всей истории до месяца *t*, минимум 36 наблюдений;
5. сохраняет `voc_data.csv` и модуль `voc_data.py` — they are used by the backtest and by every other module.

**Выравнивание по времени.** Строка с датой *t*: предикторы $G_t$ (известны на конец месяца *t*) и цель `R_next` $= R_{t+1}$. Для окна $T=12$ и прогноза на месяц $t+1$: обучаемся на строках $t-12,\dots,t-1$, прогнозируем по строке $t$.

## 1. Модуль с функциями
Ячейка записывает `voc_data.py` на диск, чтобы его можно было импортировать (и скопировать в репозиторий).

In [ ]:
%%writefile voc_data.py
"""
voc_data.py — данные для репликации Kelly, Malamud, Zhou (2024)
"The Virtue of Complexity in Return Prediction", Journal of Finance 79(1).

Что делает модуль (Section V.A статьи + сноски 33–34):
  1. Загружает месячный лист датасета Goyal–Welch (сайт Amit Goyal).
  2. Строит 15 предикторов из сноски 33:
       dfy, infl, svar, de, lty, tms, tbl, dfr, dp, dy, ltr, ep, b/m, ntis
       + один лаг рыночной доходности (mkt_lag).
  3. Строит excess return CRSP value-weighted: R_t = CRSP_SPvw_t - Rfree_t.
  4. Стандартизирует всё ТОЛЬКО по прошлому (без look-ahead):
       * доходность: делится на trailing 12-month uncentered std
             vol_t = sqrt( mean(R_{t-12}^2, ..., R_{t-1}^2) )
       * каждый предиктор: делится на expanding std по истории до t включительно,
         минимум 36 наблюдений.
  5. Выравнивает по времени: строка с датой t содержит предикторы G_t
     (известны на конец месяца t) и цель R_next = стандартизированная R_{t+1}.

Использование:
    from voc_data import load_voc_data, PREDICTORS
    df = load_voc_data()                      # скачает свежий файл Goyal
    df = load_voc_data("PredictorData2021.xlsx", end="2020-12")
    G = df[PREDICTORS].values                 # (N, 15)
    R = df["R_next"].values                   # (N,)  цель для месяца t+1

Для обучения на окне T=12 и прогноза на месяц t:
    train: строки t-12 ... t-1   (G_s -> R_next_s = R_{s+1}, все R_{s+1} <= R_t известны)
    test : строка  t             (G_t -> прогноз R_{t+1})
"""

from __future__ import annotations

import io
import warnings

import numpy as np
import pandas as pd

# Ссылка со страницы Amit Goyal (https://sites.google.com/view/agoyal145),
# "Updated data (up to 2025)". Если Goyal обновит файл — ID может поменяться.
GOYAL_SHEET_ID = "1qwpl2R_DNujpU5YUkk8lacP1tTeMb9iJ"
GOYAL_URL = f"https://docs.google.com/spreadsheets/d/{GOYAL_SHEET_ID}/export?format=xlsx"

# 15 предикторов (порядок как в сноске 33)
PREDICTORS = [
    "dfy", "infl", "svar", "de", "lty", "tms", "tbl", "dfr",
    "dp", "dy", "ltr", "ep", "bm", "ntis", "mkt_lag",
]

# Возможные названия колонок в разных версиях файла Goyal
_ALIASES = {
    "yyyymm": ["yyyymm", "date"],
    "Index": ["Index", "index", "price", "sp_index"],
    "D12": ["D12", "d12"],
    "E12": ["E12", "e12"],
    "bm": ["b/m", "bm", "b_m"],
    "tbl": ["tbl"],
    "AAA": ["AAA", "aaa"],
    "BAA": ["BAA", "baa"],
    "lty": ["lty"],
    "ntis": ["ntis"],
    "Rfree": ["Rfree", "rfree", "rf"],
    "infl": ["infl"],
    "ltr": ["ltr"],
    "corpr": ["corpr"],
    "svar": ["svar"],
    "CRSP_SPvw": ["CRSP_SPvw", "crsp_spvw", "vwret"],
}


# ----------------------------------------------------------------------------
# 1. Загрузка
# ----------------------------------------------------------------------------
def read_goyal_raw(source: str | None = None) -> pd.DataFrame:
    """Читает лист 'Monthly'. source: путь к .xlsx/.csv, URL или None (скачать)."""
    if source is None:
        source = GOYAL_URL
    if isinstance(source, str) and source.startswith("http"):
        import urllib.request
        with urllib.request.urlopen(source) as r:
            content = r.read()
        buf = io.BytesIO(content)
        if content[:2] != b"PK":  # не xlsx (например, html-страница с ошибкой)
            raise RuntimeError(
                "Не удалось скачать xlsx с Google Drive. Скачайте файл вручную со "
                "страницы Amit Goyal и передайте путь: load_voc_data('PredictorDataXXXX.xlsx')"
            )
        source = buf
    if isinstance(source, str) and source.lower().endswith(".csv"):
        raw = pd.read_csv(source)
    else:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            xls = pd.ExcelFile(source)
            sheet = next((s for s in xls.sheet_names if s.lower().startswith("month")),
                         xls.sheet_names[0])
            raw = pd.read_excel(xls, sheet)
    return _rename_columns(raw)


def _rename_columns(raw: pd.DataFrame) -> pd.DataFrame:
    cols = {c.strip(): c for c in raw.columns.astype(str)}
    lower = {c.lower(): c for c in cols}
    out = {}
    for std_name, aliases in _ALIASES.items():
        found = None
        for a in aliases:
            if a in cols:
                found = cols[a]; break
            if a.lower() in lower:
                found = cols[lower[a.lower()]]; break
        if found is None:
            raise KeyError(f"Не найдена колонка '{std_name}'. Есть колонки: {list(raw.columns)}")
        out[std_name] = pd.to_numeric(raw[found], errors="coerce")
    df = pd.DataFrame(out)
    df = df.dropna(subset=["yyyymm"])
    df["yyyymm"] = df["yyyymm"].astype(int)
    df.index = pd.PeriodIndex(
        [pd.Period(year=y // 100, month=y % 100, freq="M") for y in df["yyyymm"]], name="date"
    )
    return df.drop(columns="yyyymm").sort_index()


# ----------------------------------------------------------------------------
# 2. Предикторы (определения как в Goyal & Welch 2008)
# ----------------------------------------------------------------------------
def build_predictors(raw: pd.DataFrame, lag_infl: bool = False) -> pd.DataFrame:
    """15 сырых (нестандартизованных) предикторов, известных на конец месяца t."""
    g = pd.DataFrame(index=raw.index)
    log_p, log_d, log_e = np.log(raw["Index"]), np.log(raw["D12"]), np.log(raw["E12"])

    g["dfy"] = raw["BAA"] - raw["AAA"]                         # default yield spread
    g["infl"] = raw["infl"].shift(1) if lag_infl else raw["infl"]  # см. сноску 33
    g["svar"] = raw["svar"]                                    # stock variance
    g["de"] = log_d - log_e                                    # dividend payout ratio
    g["lty"] = raw["lty"]                                      # long-term yield
    g["tms"] = raw["lty"] - raw["tbl"]                         # term spread
    g["tbl"] = raw["tbl"]                                      # T-bill rate
    g["dfr"] = raw["corpr"] - raw["ltr"]                       # default return spread
    g["dp"] = log_d - log_p                                    # dividend-price
    g["dy"] = log_d - log_p.shift(1)                           # dividend yield
    g["ltr"] = raw["ltr"]                                      # long-term bond return
    g["ep"] = log_e - log_p                                    # earnings-price
    g["bm"] = raw["bm"]                                        # book-to-market
    g["ntis"] = raw["ntis"]                                    # net equity expansion
    g["mkt_lag"] = raw["CRSP_SPvw"] - raw["Rfree"]             # excess return месяца t
    return g[PREDICTORS]


def excess_return(raw: pd.DataFrame) -> pd.Series:
    """Excess return CRSP VW: R_t = CRSP_SPvw_t - Rfree_t."""
    return (raw["CRSP_SPvw"] - raw["Rfree"]).rename("R_raw")


# ----------------------------------------------------------------------------
# 3. Стандартизация (только прошлое!)
# ----------------------------------------------------------------------------
def standardize_returns(r: pd.Series, window: int = 12) -> pd.DataFrame:
    """R_t / sqrt(mean(R_{t-window..t-1}^2)): uncentered std по прошлым 12 месяцам (сноска 34)."""
    vol = np.sqrt((r ** 2).shift(1).rolling(window, min_periods=window).mean())
    return pd.DataFrame({"R_raw": r, "R_vol": vol, "R": r / vol})


def standardize_predictors(g: pd.DataFrame, min_periods: int = 36) -> pd.DataFrame:
    """G_t / std(G_start..G_t): expanding std по всей истории до t включительно, >= 36 наблюдений."""
    sd = g.expanding(min_periods=min_periods).std()
    return g / sd


# ----------------------------------------------------------------------------
# 4. Всё вместе
# ----------------------------------------------------------------------------
def load_voc_data(
    source: str | None = None,
    start: str | None = None,
    end: str | None = "2020-12",
    min_periods: int = 36,
    ret_window: int = 12,
    lag_infl: bool = False,
) -> pd.DataFrame:
    """
    Готовая таблица для бэктеста. Индекс — месяц t (PeriodIndex).
    Колонки:
        15 предикторов PREDICTORS (стандартизованы, известны на конец t)
        R       — стандартизированная excess return месяца t
        R_next  — стандартизированная excess return месяца t+1  (ЦЕЛЬ прогноза)
        R_raw, R_vol, R_next_raw — сырые величины для справки
    end="2020-12" как в статье (1926–2020); end=None — все доступные данные.
    Последний месяц end — это последний месяц, для которого известна цель R_next.
    """
    raw = read_goyal_raw(source)
    g = standardize_predictors(build_predictors(raw, lag_infl), min_periods)
    r = standardize_returns(excess_return(raw), ret_window)

    df = g.join(r)
    df["R_next"] = df["R"].shift(-1)
    df["R_next_raw"] = df["R_raw"].shift(-1)

    df = df.dropna(subset=PREDICTORS + ["R", "R_next"])
    if start is not None:
        df = df[df.index >= pd.Period(start, "M")]
    if end is not None:
        df = df[df.index <= pd.Period(end, "M")]
    return df


def check_no_lookahead(source: str | None = None, cut: str = "1990-12") -> bool:
    """Проверка: если обрезать исходные данные на месяце cut, значения до cut не меняются."""
    raw = read_goyal_raw(source)
    full = standardize_predictors(build_predictors(raw)).join(standardize_returns(excess_return(raw)))
    part_raw = raw[raw.index <= pd.Period(cut, "M")]
    part = standardize_predictors(build_predictors(part_raw)).join(standardize_returns(excess_return(part_raw)))
    a, b = full.loc[part.index], part
    return bool(np.allclose(a.values, b.values, equal_nan=True))


if __name__ == "__main__":
    import sys
    src = sys.argv[1] if len(sys.argv) > 1 else None
    df = load_voc_data(src)
    print(df.shape, df.index[0], "->", df.index[-1])
    print(df[PREDICTORS + ["R_next"]].describe().T.round(3))
    print("no look-ahead:", check_no_lookahead(src))

## 2. Загрузка данных
По умолчанию файл скачивается напрямую с Google Drive Amit Goyal (ссылка «Updated data» на https://sites.google.com/view/agoyal145).
Если скачивание не сработало, скачайте `.xlsx` вручную, загрузите в Colab (иконка папки слева) и укажите путь в `SOURCE`.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import importlib, voc_data; importlib.reload(voc_data)
from voc_data import load_voc_data, read_goyal_raw, check_no_lookahead, PREDICTORS

SOURCE = None          # None = скачать; или 'PredictorData2023.xlsx'
END = '2020-12'        # как в статье (1926–2020). None = все доступные данные

raw = read_goyal_raw(SOURCE)
print('Сырые данные:', raw.index[0], '->', raw.index[-1], raw.shape)
raw.tail(3)

## 3. Готовая таблица

In [ ]:
df = load_voc_data(SOURCE, end=END)
print(f'Строк: {len(df)},  с {df.index[0]} по {df.index[-1]}')
print('Первый out-of-sample прогноз при T=12 — на месяц', df.index[12] + 1)
df.head()

In [ ]:
df[PREDICTORS + ['R', 'R_next']].describe().T.round(2)

## 4. Проверки
* **Нет look-ahead:** если обрезать сырые данные на 1990-12, значения до 1990-12 не должны измениться.
* **Нет пропусков** в итоговой таблице.
* **Стандартизированная доходность** должна иметь std ≈ 1 (волатильность кластеризуется, поэтому чуть больше 1 — это нормально).

In [ ]:
print('Нет look-ahead:', check_no_lookahead(SOURCE))
print('Пропусков:', int(df.isna().sum().sum()))
print('std сырой excess return (мес.):', round(df.R_raw.std(), 4))
print('std стандартизированной R:     ', round(df.R.std(), 3))

## 5. Графики

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
t = df.index.to_timestamp()
ax[0].plot(t, df.R_raw, lw=0.6, color='#555')
ax[0].set_title('Сырая excess return CRSP VW (месячная)')
ax[1].plot(t, df.R, lw=0.6, color='#1f5fa8')
ax[1].set_title('После деления на trailing 12-month uncentered std')
for a in ax: a.axhline(0, color='k', lw=0.5); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(13, 12), sharex=True)
for a, c in zip(axes.flat, PREDICTORS):
    a.plot(t, df[c], lw=0.7, color='#1f5fa8'); a.set_title(c, fontsize=10); a.grid(alpha=0.3)
fig.suptitle('15 предикторов после expanding-std стандартизации', y=1.0)
plt.tight_layout(); plt.show()

## 6. Быстрая проверка, что данные «работают» (не часть моей задачи, просто sanity check)
Упрощённая версия главного результата статьи: RFF с $P=12\,000$, $\gamma=2$, окно $T=12$, ridge $z=10^3$, 3 сида.
В статье Sharpe такой timing-стратегии ≈ 0.47 (рис. 7). The full backtest over the grid of $P$ and $z$ is in `backtest/`.

In [ ]:
G, R = df[PREDICTORS].values, df['R_next'].values
N, T = len(R), 12

def voc_sharpe(P=12000, z=1e3, seed=0, gamma=2.0):
    rng = np.random.default_rng(seed)
    W = rng.standard_normal((G.shape[1], P // 2))
    X = gamma * G @ W
    S = np.hstack([np.sin(X), np.cos(X)])                     # ур. (20)
    pred = np.full(N, np.nan)
    for t in range(T, N):
        Str = S[t-T:t]; sd = Str.std(0) + 1e-12               # сноска 39
        Str = Str / sd
        beta = Str.T @ np.linalg.solve(z*T*np.eye(T) + Str @ Str.T, R[t-T:t])  # ridge в дуальной форме
        pred[t] = (S[t] / sd) @ beta
    m = ~np.isnan(pred); strat = pred[m] * R[m]                # π_t R_{t+1}
    return strat.mean() / strat.std() * np.sqrt(12)

srs = [voc_sharpe(seed=s) for s in range(3)]
print('Годовой Sharpe timing-стратегии:', np.round(srs, 3), '→ среднее', round(np.mean(srs), 3))

## 7. Сохранение для команды

In [ ]:
out = df.copy(); out.index = out.index.astype(str)
out.to_csv('voc_data.csv')
print('Сохранено voc_data.csv', out.shape)
# В Colab скачать: from google.colab import files; files.download('voc_data.csv'); files.download('voc_data.py')

### Как использовать остальным
```python
import pandas as pd
df = pd.read_csv('voc_data.csv', index_col='date')
PREDICTORS = ['dfy','infl','svar','de','lty','tms','tbl','dfr','dp','dy','ltr','ep','bm','ntis','mkt_lag']
G = df[PREDICTORS].values   # (N, 15)  предикторы на конец месяца t
R = df['R_next'].values     # (N,)     стандартизированная excess return месяца t+1
```
**Выбор, который мы сделали (стоит упомянуть на слайде Critique):** vol для доходности считается по месяцам $t-12..t-1$ (без текущего); `mkt_lag` — сырая excess return месяца $t$, стандартизированная как остальные предикторы; инфляция без дополнительного лага (конвенция Goyal–Welch–Zafirov, сноска 33; `lag_infl=True` сдвигает её на месяц).